In [16]:
import json
import ollama
from dataclasses import dataclass


@dataclass
class Config:
    llm_model: str = "llama3.1:8b"
    ollama_host: str = "http://localhost:11434"
    temperature: float = 0.0


config = Config()
client = ollama.Client(host=config.ollama_host)


def llm_json(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat(
        model=config.llm_model, messages=messages,
        format="json", options={"temperature": config.temperature},
    )
    raw = resp["message"]["content"]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        return json.loads(raw[start: end + 1])


def llm_text(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat(
        model=config.llm_model, messages=messages,
        options={"temperature": config.temperature},
    )
    return resp["message"]["content"].strip()


# Sanity check
try:
    client.list()
    print("Ollama conectado en", config.ollama_host)
    print(f"Modelo: {config.llm_model}")
except Exception as e:
    print("Ollama no responde:", e)

Ollama conectado en http://localhost:11434
Modelo: llama3.1:8b


In [17]:
# Extractor de entidades

ENTITY_EXTRACTOR_PROMPT = """You are an entity extractor for a long-term memory system.
The speaker is: {speaker}

Any first-person pronoun (I, me, my, mine, we, our) refers to {speaker}.
Always include {speaker} as a "person" entity when first-person pronouns appear.

Identify the key entities mentioned. For each, output:
- "name": canonical lowercase identifier with underscores (no spaces, no articles).
- "type": one of [person, location, organization, object, concept, event, date, attribute].

Naming convention examples:
- "the Eiffel Tower" → "eiffel_tower"
- "Dr. Maria Garcia" → "maria_garcia" 
- "Microsoft Office" → "microsoft_office"
- "my younger brother" (if named John) → "john"
- "my younger brother" (if unnamed) → "speaker_brother"

What counts as an entity:
- Specific, named, persistent: people, places, organizations, products, named events.
- DO include: specific names, brands, locations, job titles, medical conditions, identified relationships.
- DO NOT include: generic concepts ("food", "happiness"), vague references ("things", "stuff"), 
  filler words, or anything not specifically identified.

OUTPUT FORMAT — return ONLY this JSON:
{{"entities": [{{"name": "<name>", "type": "<type>"}}, ...]}}

If nothing relevant: {{"entities": []}}.

CONVERSATION:
---
{conversation}
---
"""


def extract_entities(text, speaker):
    """Extrae entidades del texto."""
    prompt = ENTITY_EXTRACTOR_PROMPT.format(conversation=text, speaker=speaker)
    result = llm_json(prompt)
    return result.get("entities", [])

In [18]:
# Extractor de resumen y tópicos
SUMMARY_PROMPT = """You are a memory system processing a conversation turn.

Generate a 1-2 sentence summary (third-person, max ~30 words) that PRESERVES all 
specific factual values mentioned: numbers, money, times, dates, names, brands, 
durations, model numbers, identifying details.

A good summary is a factual record, not a vague description.

Examples:

  Original: "I just got a 2020 Honda Civic for $18,500 last weekend."
  GOOD: "The user bought a 2020 Honda Civic for $18,500 last weekend."
  BAD:  "The user mentioned a car purchase."

  Original: "On Mondays I have therapy at 4pm with Dr. Lopez."
  GOOD: "The user has therapy with Dr. Lopez at 4pm on Mondays."
  BAD:  "The user described their therapy schedule."

Also extract 2-5 short lowercase topics.

Speaker: {speaker}

OUTPUT FORMAT — return ONLY this JSON:
{{"summary": "<factual summary preserving specific values>", "topics": ["<t1>", "<t2>", ...]}}

CONVERSATION TURN:
---
{turn}
---
"""


def extract_summary_topics(text, speaker):
    """Extrae resumen breve + tópicos clave del turno."""
    prompt = SUMMARY_PROMPT.format(turn=text, speaker=speaker)
    result = llm_json(prompt)
    return {
        "summary": result.get("summary", text[:80]),
        "topics": result.get("topics", []),
    }

In [19]:
# FASE 1

def phase1_extract_c1(text, speaker):
    """Fase 1 (Opción C.1): solo entidades + summary + topics.
    NO extrae relaciones — eso es lo que diferencia de Mem0g.
    """
    entities = extract_entities(text, speaker=speaker)
    # Garantizar que el speaker esté como entidad
    if not any(e["name"] == speaker for e in entities):
        entities.append({"name": speaker, "type": "person"})
    
    summary_topics = extract_summary_topics(text, speaker=speaker)
    
    return {
        "entities": entities,
        "summary": summary_topics["summary"],
        "topics": summary_topics["topics"],
    }


print("Fase 1")

Fase 1


In [20]:
test_turns = [
    "I live in San Francisco with my partner Sam.",
    "My favorite coffee shop is Sightglass and I go there every morning before work.",
    "I just moved from Seattle to New York for a new job at Stripe as a software engineer.",
    "Yesterday I went to a LGBTQ support group and it was really powerful.",
    "I have two kids, Ava and Noah, and they keep me busy.",
]

for i, turn in enumerate(test_turns, 1):
    print(f"\n{'='*72}")
    print(f"TURNO {i}: {turn}")
    print('='*72)
    
    result = phase1_extract_c1(turn, speaker="user")
    
    print(f"\n  RESUMEN:  {result['summary']}")
    print(f"  TÓPICOS:  {result['topics']}")
    print(f"\n  ENTIDADES ({len(result['entities'])}):")
    for e in result['entities']:
        print(f"    - {e['name']:30s} ({e['type']})")


TURNO 1: I live in San Francisco with my partner Sam.

  RESUMEN:  The user lives in San Francisco with their partner Sam.
  TÓPICOS:  ['location', 'relationship']

  ENTIDADES (3):
    - user                           (person)
    - san_francisco                  (location)
    - sam                            (person)

TURNO 2: My favorite coffee shop is Sightglass and I go there every morning before work.

  RESUMEN:  The user's favorite coffee shop is Sightglass, which they visit daily.
  TÓPICOS:  ['coffee', 'shop']

  ENTIDADES (2):
    - sightglass                     (organization)
    - user                           (person)

TURNO 3: I just moved from Seattle to New York for a new job at Stripe as a software engineer.

  RESUMEN:  The user moved from Seattle to New York for a job at Stripe.
  TÓPICOS:  ['move', 'job']

  ENTIDADES (5):
    - user_job                       (attribute)
    - seattle                        (location)
    - new_york                       (locat

In [21]:
import math
import networkx as nx
from datetime import datetime, timezone


def now_iso():
    return datetime.now(timezone.utc).isoformat()


class ConvMemoryGraph:

    
    def __init__(self,
                 alpha=0.3, beta=0.4, gamma=0.3, lam=0.3, n_max=30,
                 exclude_speaker_from_relevance=True):
        self.g = nx.MultiDiGraph()
        self.turn_counter = 0
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lam = lam
        self.n_max = n_max

        self.exclude_speaker_from_relevance = exclude_speaker_from_relevance
    
    # Fase 2
    
    def add(self, text, speaker="user"):

        # FASE 1
        extracted = phase1_extract_c1(text, speaker)
        entities = extracted["entities"]
        summary = extracted["summary"]
        topics = extracted["topics"]
        
        # insertar nodo de turno
        position = self.turn_counter
        turn_id = f"t{position}"
        self.g.add_node(
            turn_id,
            node_type="turn",
            position=position,
            summary=summary,
            topics=topics,
            role="user",
            r=1.0,
            created_at=now_iso(),
        )
        
        #añadir/actualizar entidades + MENTIONS
        for e in entities:
            name = e["name"]
            etype = e["type"]
            if name not in self.g.nodes:
                self.g.add_node(
                    name,
                    node_type="entity",
                    entity_type=etype,
                    attributes={},
                    first_seen=position,
                    last_seen=position,
                    is_speaker=(name == speaker),
                    created_at=now_iso(),
                )
            else:
                self.g.nodes[name]["last_seen"] = position
            
            self.g.add_edge(turn_id, name,
                            edge_type="MENTIONS",
                            created_at=now_iso())
        
        # recomputar r(t_i) para todos los turnos
        t_actual = position
        for tid in self._turn_ids():
            self.g.nodes[tid]["r"] = self._compute_r(tid, t_actual)
        
        # poda condicional
        n_pruned = 0
        if self._n_turns() > self.n_max:
            n_pruned = self._prune_low_relevance_turns()
        
        self.turn_counter += 1
        return {
            "turn_id": turn_id,
            "position": position,
            "n_entities": len(entities),
            "n_pruned": n_pruned,
        }
    
    # r(t_i) 
    
    def _compute_r(self, turn_id, t_actual):
        """Calcula r(t_i) = α·ant(t_i) + β·men(t_i) + γ·ult(t_i)."""
        i = self.g.nodes[turn_id]["position"]
        
        #  antigüedad 
        ant = math.exp(-self.lam * (t_actual - i))
        
        # menciones 
        entities_i = self._entities_of_turn(turn_id)
        sharing_count = 0
        last_mention_j = i
        
        for tid in self._turn_ids():
            j = self.g.nodes[tid]["position"]
            if j < i:
                continue
            if j == i:
                sharing_count += 1  # auto-mención
                continue
            entities_j = self._entities_of_turn(tid)
            if entities_i & entities_j:
                sharing_count += 1
                last_mention_j = max(last_mention_j, j)
        
        men = sharing_count / (t_actual - i + 1)
        
        # recencia
        ult = math.exp(-self.lam * (t_actual - last_mention_j))
        
        return self.alpha * ant + self.beta * men + self.gamma * ult
    

    
    def _turn_ids(self):
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "turn"]
    
    def _entity_ids(self):
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "entity"]
    
    def _n_turns(self):
        return len(self._turn_ids())
    
    def _entities_of_turn(self, turn_id, exclude_speaker=None):
        if exclude_speaker is None:
            exclude_speaker = self.exclude_speaker_from_relevance
        out = set()
        for _, v, d in self.g.out_edges(turn_id, data=True):
            if d.get("edge_type") != "MENTIONS":
                continue
            if exclude_speaker and self.g.nodes[v].get("is_speaker"):
                continue
            out.add(v)
        return out
    
    def _prune_low_relevance_turns(self):
        n_to_remove = self._n_turns() - self.n_max
        if n_to_remove <= 0:
            return 0
        turns_sorted = sorted(self._turn_ids(),
                              key=lambda tid: self.g.nodes[tid]["r"])
        for tid in turns_sorted[:n_to_remove]:
            self.g.remove_node(tid)
        return n_to_remove
    
    
    def show_state(self):
        """Imprime el estado del grafo de forma legible."""
        print(f"\n{'='*72}")
        print(f"ESTADO DEL GRAFO  |  turnos: {self._n_turns()}  |  entidades: {len(self._entity_ids())}")
        print('='*72)
        
        print(f"\nNODOS DE TURNO:")
        for tid in sorted(self._turn_ids(),
                          key=lambda x: self.g.nodes[x]["position"]):
            d = self.g.nodes[tid]
            menciona = sorted(self._entities_of_turn(tid, exclude_speaker=False))
            bar = "█" * int(d["r"] * 20) + "·" * (20 - int(d["r"] * 20))
            print(f"  [{tid}] pos={d['position']}  r={d['r']:.3f}  {bar}")
            print(f"        resumen:  {d['summary']}")
            print(f"        tópicos:  {d['topics']}")
            print(f"        menciona: {menciona}")
            print()
        
        print(f"NODOS DE ENTIDAD:")
        for eid in sorted(self._entity_ids()):
            d = self.g.nodes[eid]
            mentioning = sorted([u for u, v, td
                                  in self.g.in_edges(eid, data=True)
                                  if td.get("edge_type") == "MENTIONS"])
            flag = " (speaker)" if d.get("is_speaker") else ""
            print(f"  [{eid}]{flag}  tipo={d['entity_type']}  "
                  f"first=t{d['first_seen']}, last=t{d['last_seen']}  "
                  f"mencionada por: {mentioning}")




In [22]:
mem = ConvMemoryGraph(alpha=0.3, beta=0.4, gamma=0.3, lam=0.3, n_max=30)

dialog = [
    "I live in San Francisco with my partner Sam.",
    "My favorite coffee shop is Sightglass.",
    "I just moved to New York for a new job at Stripe.",
    "I had dinner with Sam at our favorite restaurant.",
]

print("INGESTA DEL DIÁLOGO\n" + "="*72)
for i, turn in enumerate(dialog):
    print(f"\n>>> Turno {i}: '{turn}'")
    result = mem.add(turn, speaker="user")
    print(f"    → {result}")

mem.show_state()

INGESTA DEL DIÁLOGO

>>> Turno 0: 'I live in San Francisco with my partner Sam.'
    → {'turn_id': 't0', 'position': 0, 'n_entities': 3, 'n_pruned': 0}

>>> Turno 1: 'My favorite coffee shop is Sightglass.'
    → {'turn_id': 't1', 'position': 1, 'n_entities': 3, 'n_pruned': 0}

>>> Turno 2: 'I just moved to New York for a new job at Stripe.'
    → {'turn_id': 't2', 'position': 2, 'n_entities': 3, 'n_pruned': 0}

>>> Turno 3: 'I had dinner with Sam at our favorite restaurant.'
    → {'turn_id': 't3', 'position': 3, 'n_entities': 3, 'n_pruned': 0}

ESTADO DEL GRAFO  |  turnos: 4  |  entidades: 8

NODOS DE TURNO:
  [t0] pos=0  r=0.622  ████████████········
        resumen:  The user lives in San Francisco with their partner Sam.
        tópicos:  ['location', 'relationship']
        menciona: ['sam', 'san_francisco', 'user']

  [t1] pos=1  r=0.463  █████████···········
        resumen:  The user mentioned their favorite coffee shop as Sightglass.
        tópicos:  ['coffee', 'shop']
     

In [23]:
ANSWER_PROMPT_C1 = """You are a memory assistant answering questions about a user.

The MEMORIES below are summaries of past turns, sorted in REVERSE CHRONOLOGICAL ORDER.

ANSWER RULES:
1. Be CONCISE — output only the answer, no preamble.
2. Match the answer to the question's intent:
   - "Where does X live?" → output a geographic place (city, country, address).
   - "Where does X work?" → output an EMPLOYER (company, organization), NOT a city.
   - "Where did X buy ...?" → output a store or location of purchase.
   - "Who ..." → output a person's name.
   - "How many / how much ..." → output a number or amount.
   - "When ..." → output a date or time.
3. RECENCY: if memories conflict, trust the more recent (higher position).
4. Read carefully: a memory like "moved to City for a job at Company" means:
   - "lives in" → City
   - "works at" → Company  
5. Only say "I don't know" if the memories truly contain no relevant info.

MEMORIES (MOST RECENT FIRST):
{context}

QUESTION: {query}

ANSWER:"""


def select_top_k_turns(mem, k=5):
    """Selecciona los k turnos con mayor r(t_i)."""
    turns_sorted = sorted(
        mem._turn_ids(),
        key=lambda tid: mem.g.nodes[tid]["r"],
        reverse=True,
    )
    return turns_sorted[:k]


def build_context(mem, turn_ids):
    sorted_by_position = sorted(
        turn_ids,
        key=lambda tid: mem.g.nodes[tid]["position"],
        reverse=True,
    )
    
    lines = []
    for tid in sorted_by_position:
        d = mem.g.nodes[tid]
        lines.append(f"[Turn {d['position']}, r={d['r']:.2f}] {d['summary']}")
        if d.get("topics"):
            lines.append(f"  Topics: {', '.join(d['topics'])}")
        mentions = sorted(mem._entities_of_turn(tid, exclude_speaker=False))
        if mentions:
            ents_str = ", ".join(
                f"{m} ({mem.g.nodes[m].get('entity_type', '?')})"
                for m in mentions
            )
            lines.append(f"  Mentions: {ents_str}")
        lines.append("")
    return "\n".join(lines).strip()


def answer(mem, query, k=5, verbose=False):
    """Genera respuesta a una consulta usando top-k turnos por r(t_i)."""
    top_turns = select_top_k_turns(mem, k=k)
    if not top_turns:
        return "I don't know — no memories available yet."
    
    context = build_context(mem, top_turns)
    prompt = ANSWER_PROMPT_C1.format(context=context, query=query)
    
    if verbose:
        print("=== CONTEXTO ENVIADO AL LLM ===")
        print(context)
        print(f"\nQUESTION: {query}\n")
        print("=" * 50)
    
    return llm_text(prompt)


In [24]:
queries = [
    "Where does the user live now?",
    "Who is the user's partner?",
    "Where does the user work?",
    "What is the user's favorite coffee shop?",
    "Where did the user have dinner recently?",
]

for q in queries:
    print(f"\n{'='*72}")
    print(f"Q: {q}")
    print('='*72)
    ans = answer(mem, q, k=5)
    print(f"A: {ans}")


Q: Where does the user live now?
A: New York

Q: Who is the user's partner?
A: Sam

Q: Where does the user work?
A: Stripe

Q: What is the user's favorite coffee shop?
A: Sightglass

Q: Where did the user have dinner recently?
A: user_favourite_restaurant


In [25]:
ans = answer(mem, "Where does the user live now?", k=5, verbose=True)
print(f"\nRESPUESTA FINAL: {ans}")

=== CONTEXTO ENVIADO AL LLM ===
[Turn 3, r=1.00] The user had dinner with Sam at their favorite restaurant.
  Topics: dinner, restaurant
  Mentions: sam (person), user (person), user_favourite_restaurant (location)

[Turn 2, r=0.64] The user moved to New York for a job at Stripe.
  Topics: moving, new job
  Mentions: new_york (location), stripe (organization), user (person)

[Turn 1, r=0.46] The user mentioned their favorite coffee shop as Sightglass.
  Topics: coffee, shop
  Mentions: sightglass (organization), speaker_favorite_coffee_shop (location), user (person)

[Turn 0, r=0.62] The user lives in San Francisco with their partner Sam.
  Topics: location, relationship
  Mentions: sam (person), san_francisco (location), user (person)

QUESTION: Where does the user live now?


RESPUESTA FINAL: New York
